# PD Voice Pipeline on Google Colab

This notebook clones the repository, installs dependencies, and runs the pipeline in Colab.

In [ ]:
# 1) Clone repository
REPO_URL = "https://github.com/<your-user>/<your-repo>.git"  # change this
REPO_DIR = "Pipeline-Implementation"

!rm -rf $REPO_DIR
!git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!ls

In [ ]:
# 2) Install dependencies
!pip -q install --upgrade pip
!pip -q install -r Wav2Vec2/requirements.txt openpyxl umap-learn

# Optional: check GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3) Provide data files
You need these files/folders inside the repo root:
- `final_selected.xlsx`
- raw audio folder(s) matching `mpower_voice_data_flac*`

If your data is in Google Drive, mount and copy it below.

In [ ]:
# 4) Mount Drive (optional) and copy dataset
from google.colab import drive
drive.mount('/content/drive')

# Update these paths for your Drive
DRIVE_ROOT = '/content/drive/MyDrive/PD_data'

!cp -f "$DRIVE_ROOT/final_selected.xlsx" ./final_selected.xlsx
!cp -r "$DRIVE_ROOT/mpower_voice_data_flac-20260127T044147Z-3-012" ./
!ls

In [ ]:
# 5) Run FULL mode (0s-10s segmentation + embeddings + analysis)
import torch
GPU_ARG = '--gpu 0' if torch.cuda.is_available() else ''

# Balanced cap per class: N means PD=N and HC=N across the full flow.
NUM_IMAGES_PER_CLASS = 1000
SEED = 42

!python audio_segmentation.py --mode full --max-per-class $NUM_IMAGES_PER_CLASS --seed $SEED
!python Wav2Vec2/pipeline.py --mode full $GPU_ARG --max-per-class $NUM_IMAGES_PER_CLASS --seed $SEED
!python HuBERT/pipeline.py --mode full $GPU_ARG --max-per-class $NUM_IMAGES_PER_CLASS --seed $SEED
!python comparative_analysis.py --mode full --n-trials 20 --final-epochs 100 --max-per-class $NUM_IMAGES_PER_CLASS --seed $SEED

In [ ]:
# 6) (Optional) Run SEGMENT mode instead
# NUM_IMAGES_PER_CLASS = 1000
# SEED = 42
# !python audio_segmentation.py --mode segment --max-per-class $NUM_IMAGES_PER_CLASS --seed $SEED
# !python Wav2Vec2/pipeline.py --mode segment $GPU_ARG --max-per-class $NUM_IMAGES_PER_CLASS --seed $SEED
# !python HuBERT/pipeline.py --mode segment $GPU_ARG --max-per-class $NUM_IMAGES_PER_CLASS --seed $SEED
# !python comparative_analysis.py --mode segment --n-trials 20 --final-epochs 100 --max-per-class $NUM_IMAGES_PER_CLASS --seed $SEED

In [ ]:
# 7) Show output artifacts
!find results -maxdepth 4 -type f | head -n 100